# Entraînement final et exploration du modèle BERTopic

Ce notebook recharge les paramètres retenus lors de l'optimisation, entraîne le modèle final et explore la structure thématique du corpus.

In [9]:
import gc
import time
import warnings
from itertools import combinations
from pathlib import Path #pour la gestion des chemins de fichiers
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP
from hdbscan import HDBSCAN
from hdbscan.validity import validity_index

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer #pour les embeddings de phrase

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

COL_TEXTE = "phrases_lemm"
COL_ROMAN = "roman"

warnings.filterwarnings("ignore")

## 1. Chargement du corpus et chargement ou calcul des embeddings

In [12]:
# Racine du projet ZOLA-DTM
PROJECT_ROOT = Path.cwd().parent

# Dossiers principaux
DATA_DIR = PROJECT_ROOT / "data"
DONNEES_ANNEX_DIR = DATA_DIR / "donnees_annex"
CACHE_DIR = DONNEES_ANNEX_DIR / "data_cache"

# Fichier des embeddings
CHEMIN_EMBEDDINGS = CACHE_DIR / "embeddings_sentence_camembert.npy"

print("Répertoire courant :", Path.cwd())
print("Racine projet       :", PROJECT_ROOT)
print("Dossier data        :", DATA_DIR)
print("Dossier cache       :", CACHE_DIR)
print("Embeddings          :", CHEMIN_EMBEDDINGS)

print("==========================================")
print("Vérification de l'existence des fichiers :")
chemin_csv = Path("../data/2_processed/02_corpus_zola_lematise_128.csv")
print("cwd =", Path.cwd())
print("chemin_csv =", chemin_csv)
print("resolve =", chemin_csv.resolve())
print("exists =", chemin_csv.exists())

Répertoire courant : /Users/morganr/zola-dtm/02_notebook_analyse
Racine projet       : /Users/morganr/zola-dtm
Dossier data        : /Users/morganr/zola-dtm/data
Dossier cache       : /Users/morganr/zola-dtm/data/donnees_annex/data_cache
Embeddings          : /Users/morganr/zola-dtm/data/donnees_annex/data_cache/embeddings_sentence_camembert.npy
Vérification de l'existence des fichiers :
cwd = /Users/morganr/zola-dtm/02_notebook_analyse
chemin_csv = ../data/2_processed/02_corpus_zola_lematise_128.csv
resolve = /Users/morganr/zola-dtm/data/2_processed/02_corpus_zola_lematise_128.csv
exists = True


In [13]:
df = pd.read_csv(chemin_csv, encoding="utf-8")

CHEMIN_EMBEDDINGS = Path("../data/donnees_annex/data_cache/embeddings_sentence_camembert.npy")

if CHEMIN_EMBEDDINGS.exists():
    print("Chargement des embeddings sauvegardés...")
    embeddings = np.load(CHEMIN_EMBEDDINGS, allow_pickle=False)
else:
    embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base")

    print("Génération des embeddings sémantiques...")
    embeddings = embedding_model.encode(
        df["texte"].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    CHEMIN_EMBEDDINGS.parent.mkdir(parents=True, exist_ok=True)
    np.save(CHEMIN_EMBEDDINGS, embeddings)
    print(f"Embeddings sauvegardés dans : {CHEMIN_EMBEDDINGS}")

Chargement des embeddings sauvegardés...


In [14]:
df.head(3)

,roman,annee,ordre_romans,paquet_id,texte,nb_tokens_camembert,phrases_lemm
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",105,voici hiver air matin devenir frais mettre man...
1,1865 La confession de Claude.,1865,1,2,"Mon grenier, tout au haut d’un escalier humide...",124,grenier haut escalier humide grand irrégulier ...
2,1865 La confession de Claude.,1865,1,3,"Le soir, quand le vent ébranle la porte et que...",114,soir vent ébranler porte mur vaciller flamme l...


## 2. Préparation des documents pour BERTopic

In [15]:
# Vérification des colonnes
assert COL_TEXTE in df.columns, (
    f"La colonne '{COL_TEXTE}' n'existe pas dans df."
)
assert COL_ROMAN in df.columns, (
    f"La colonne '{COL_ROMAN}' n'existe pas dans df."
)

# Conversion des embeddings
embeddings_array_initial = np.asarray(embeddings)

assert len(df) == len(embeddings_array_initial), (
    "Le nombre de lignes de df ne correspond pas "
    "au nombre d'embeddings."
)

# Masque des documents non vides
masque_documents = (df[COL_TEXTE].notna() & df[COL_TEXTE].astype(str).str.strip().ne(""))

# DataFrame utilisé par BERTopic
df_model = (df.loc[masque_documents].copy().reset_index(drop=True))

# Documents
documents = (df_model[COL_TEXTE].astype(str).tolist())

# Romans associés aux documents
romans = (df_model[COL_ROMAN].astype(str).tolist())

# Embeddings alignés
embeddings_array = embeddings_array_initial[masque_documents.to_numpy()]

# Tokenisation simple pour la cohérence C_v
texts_tokenises = [
    document.split()
    for document in documents
]

# Dictionnaire Gensim
dictionary = Dictionary(texts_tokenises)


print("Nombre de documents :", len(documents))
print("Nombre de romans :", df_model[COL_ROMAN].nunique())
print("Dimensions des embeddings :", embeddings_array.shape)
print("Taille du dictionnaire :", len(dictionary))

Nombre de documents : 61432
Nombre de romans : 31
Dimensions des embeddings : (61432, 768)
Taille du dictionnaire : 23302


## 3. Fonctions d'évaluation finale

In [16]:
def extraire_mots_topics(
    topic_model,
    labels,
    top_n_words=15,
    dictionary=None
):
    """
    Extrait les mots des topics, en excluant le topic -1.

    Si un dictionnaire Gensim est fourni, seuls les mots présents
    dans ce dictionnaire sont conservés.
    """

    labels = np.asarray(labels)

    topic_ids = sorted(
        int(topic_id)
        for topic_id in np.unique(labels)
        if topic_id != -1
    )

    topics_words = []

    for topic_id in topic_ids:

        representation = topic_model.get_topic(topic_id)

        if not representation:
            continue

        words = [
            word
            for word, _ in representation[:top_n_words]
        ]

        if dictionary is not None:
            words = [
                word
                for word in words
                if word in dictionary.token2id
            ]

        # Éviter les topics insuffisamment représentés
        if len(words) >= 2:
            topics_words.append(words)

    return topics_words


def calculer_coherence_cv(
    topic_model,
    labels,
    texts_tokenises,
    dictionary,
    top_n_words=10
):
    """
    Calcule la cohérence C_v des topics BERTopic.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=dictionary
    )

    if len(topics_words) < 2:
        return np.nan

    try:
        coherence_model = CoherenceModel(
            topics=topics_words,
            texts=texts_tokenises,
            dictionary=dictionary,
            coherence="c_v",
            topn=top_n_words,
            processes=1
        )

        return float(coherence_model.get_coherence())

    except Exception:
        return np.nan

def calculer_diversite_topics(
    topic_model,
    labels,
    top_n_words=10
):
    """
    Diversité lexicale :
    nombre de termes uniques / nombre total de termes.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=None
    )

    tous_les_mots = [
        word
        for topic_words in topics_words
        for word in topic_words
    ]

    if not tous_les_mots:
        return np.nan

    return len(set(tous_les_mots)) / len(tous_les_mots)
    

## 4. Chargement des paramètres et entraînement du modèle final

Les paramètres sélectionnés dans le notebook d'optimisation sont rechargés depuis le fichier JSON.

In [17]:
CHEMIN_PARAMETRES = Path("../data/donnees_annex/meilleurs_parametres_bertopic.json")

with CHEMIN_PARAMETRES.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

best_umap_model = UMAP(
    n_neighbors=best_params["n_neighbors"],
    n_components=best_params["n_components"],
    min_dist=0.0,
    metric="cosine",
    random_state=best_params["random_state"],
    low_memory=True
)

best_hdbscan_model = HDBSCAN(
    min_cluster_size=best_params["min_cluster_size"],
    min_samples=best_params["min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
    core_dist_n_jobs=-1
)

best_vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.8,
)

best_ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model = BERTopic(
    language="french",
    hdbscan_model=best_hdbscan_model,
    umap_model=best_umap_model,
    vectorizer_model=best_vectorizer_model,
    ctfidf_model=best_ctfidf_model,
    calculate_probabilities=False,
    nr_topics=None,
    verbose=True
)

best_topics_raw, _ = best_topic_model.fit_transform(
    documents,
    embeddings=embeddings_array
)

best_topics_raw = np.asarray(best_topics_raw)


nombre_topics_bruts = len(
    np.unique(
        best_topics_raw[
            best_topics_raw != -1
        ]
    )
)

taux_outliers_brut = np.mean(best_topics_raw == -1)

print("Nombre naturel de topics :", nombre_topics_bruts)

print( f"Taux brut d'outliers : " f"{taux_outliers_brut:.1%}")

display(best_topic_model.get_topic_info())

2026-08-17 11:58:12,471 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-17 12:00:36,443 - BERTopic - Dimensionality - Completed ✓
2026-08-17 12:00:36,461 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-17 12:00:48,651 - BERTopic - Cluster - Completed ✓
2026-08-17 12:00:48,667 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-17 12:00:49,698 - BERTopic - Representation - Completed ✓


Nombre naturel de topics : 17
Taux brut d'outliers : 76.5%


,Topic,Count,Name,Representation,Representative_Docs
0,-1,46965,-1_jambe_chaise_payer_rougon,"[jambe, chaise, payer, rougon, mariage, dîner,...",[effet express arriver vapeur passer coup long...
1,0,2729,0_horizon_arbre_vert_toiture,"[horizon, arbre, vert, toiture, façade, marbre...",[immense plan relief aller bord ciel colline t...
2,1,2665,1_justice_humanité_social_nation,"[justice, humanité, social, nation, science, c...",[voix briser chercher punition faute permettre...
3,2,1721,2_écoute_te_chéri_veu,"[écoute, te, chéri, veu, tu, laisse, tai, tien...",[celui méchant être jurer vouloir lâche fond d...
4,3,942,3_armée_prussien_général_batterie,"[armée, prussien, général, batterie, soldat, t...",[autre voix expliquer mouvement nouveau arrive...
5,4,852,4_million_bénéfice_rente_bourse,"[million, bénéfice, rente, bourse, capital, ac...",[assemblée général avoir lieu fin avril bilan ...
6,5,635,5_sommeil_oreiller_couverture_drap,"[sommeil, oreiller, couverture, drap, bougie, ...",[silence nuit sembler souffrir étendre drap bo...
7,6,625,6_coupeau_lorilleu_soupe_pomme,"[coupeau, lorilleu, soupe, pomme, plat, vin, a...",[avoir vice devenir avoir prise maman hocher t...
8,7,608,7_amant_souffrance_affection_adoration,"[amant, souffrance, affection, adoration, care...",[voir bras torturer évoquer spectacle étrange ...
9,8,534,8_comte_inviter_aimable_déjeuner,"[comte, inviter, aimable, déjeuner, comtesse, ...",[vouloir lever descendre enfant bras désireux ...


## 5. Réaffectation des documents hors topic

Plusieurs seuils sont comparés avant de retenir le seuil final de réaffectation des outliers.

In [18]:
seuils_outliers = [
    0.0,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

comparaisons_seuils = []
topics_par_seuil = {}

for seuil in seuils_outliers:
    topics_test = best_topic_model.reduce_outliers(
        documents,
        best_topics_raw,
        strategy="embeddings",
        embeddings=embeddings_array,
        threshold=seuil)

    topics_test = np.asarray(topics_test)

    topics_par_seuil[seuil] = topics_test

    masque_assignes = topics_test != -1

    tailles_topics = (pd.Series(topics_test[masque_assignes]).value_counts())

    nombre_assignes = int(masque_assignes.sum())

    nombre_reassignes = int(((best_topics_raw == -1) & (topics_test != -1)).sum())

    comparaisons_seuils.append({
        "threshold": seuil,
        "n_topics": len(tailles_topics),
        "outlier_rate_remaining": np.mean(
            topics_test == -1
        ),
        "n_reassigned": nombre_reassignes,
        "smallest_topic": (
            tailles_topics.min()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic": (
            tailles_topics.max()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic_share": (
            tailles_topics.max() / nombre_assignes
            if nombre_assignes > 0
            else np.nan
        )
    })


comparaison_seuils_df = pd.DataFrame(comparaisons_seuils)

display(
    comparaison_seuils_df.style.format({
        "threshold": "{:.2f}",
        "outlier_rate_remaining": "{:.1%}",
        "largest_topic_share": "{:.1%}"
    })
)

,threshold,n_topics,outlier_rate_remaining,n_reassigned,smallest_topic,largest_topic,largest_topic_share
0,0.00,17,0.0%,46964,1341,7589,12.4%
1,0.10,17,0.0%,46963,1341,7589,12.4%
2,0.20,17,0.0%,46952,1339,7588,12.4%
3,0.30,17,0.1%,46909,1333,7584,12.4%
4,0.40,17,0.4%,46741,1324,7572,12.4%
5,0.50,17,1.5%,46032,1290,7512,12.4%


In [19]:
SEUIL_OUTLIERS_FINAL = 0.20

topics_finaux = np.asarray(topics_par_seuil[SEUIL_OUTLIERS_FINAL])

print(
    f"Taux d'outliers final : "
    f"{np.mean(topics_finaux == -1):.1%}"
)


print(
    "Nombre final de topics :",
    len(np.unique(topics_finaux[topics_finaux != -1])))

distribution_topics = (
    pd.Series(topics_finaux)
    .value_counts()
    .sort_index()
    .rename_axis("Topic")
    .reset_index(name="Count")
)

display(distribution_topics)

Taux d'outliers final : 0.0%
Nombre final de topics : 17


,Topic,Count
0,-1,13
1,0,5397
2,1,4554
3,2,3188
4,3,2655
5,4,3817
6,5,5277
7,6,5714
8,7,4725
9,8,7588


## 6. Raffinement de la représentation lexicale des topics

In [30]:
stopwords_corpus = ["deberl", "men", "yole","embrass", "revien", "tai", "connai", "quidquid", "trouche", "hattoy", "sai", 'fasse','interrompit',
                    'rauque', 'pan', 'croyez', 'faite' ,'rêv', 'moi', 'te', 'veu', 'tu', 'fai', 'mon', "me"
                    ]

vectorizer_final = CountVectorizer(
    stop_words=stopwords_corpus,
    min_df=2,
    max_df=0.80,
    #ngram_range=(1, 2)
)

ctfidf_final = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model.update_topics(
    documents,
    topics=topics_finaux,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    top_n_words=15
)

display(best_topic_model.get_topic_info())

2026-08-17 12:29:25,354 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,13,-1_dormeur_indication_égorger_,"[dormeur, indication, égorger, , , , , , , , ,...",[effet express arriver vapeur passer coup long...
1,0,5397,0_toiture_géant_feuillage_rangée,"[toiture, géant, feuillage, rangée, aligner, n...",[immense plan relief aller bord ciel colline t...
2,1,4554,1_nation_catholicisme_pape_dogme,"[nation, catholicisme, pape, dogme, évolution,...",[voix briser chercher punition faute permettre...
3,2,3188,2_pleure_mienne_mien_ose,"[pleure, mienne, mien, ose, pense, catéchisme,...",[celui méchant être jurer vouloir lâche fond d...
4,3,2655,3_troupe_barricade_batterie_obu,"[troupe, barricade, batterie, obu, national, l...",[autre voix expliquer mouvement nouveau arrive...
5,4,3817,4_banque_capital_économie_spéculation,"[banque, capital, économie, spéculation, achat...",[assemblée général avoir lieu fin avril bilan ...
6,5,5277,5_oreiller_insomnie_recoucher_ronfler,"[oreiller, insomnie, recoucher, ronfler, séant...",[silence nuit sembler souffrir étendre drap bo...
7,6,5714,6_soupe_lorilleu_litre_gervaise,"[soupe, lorilleu, litre, gervaise, blanchisseu...",[avoir vice devenir avoir prise maman hocher t...
8,7,4725,7_épouse_maternité_implacable_virginité,"[épouse, maternité, implacable, virginité, ren...",[voir bras torturer évoquer spectacle étrange ...
9,8,7588,8_thé_malle_loge_cocher,"[thé, malle, loge, cocher, piano, vestibule, c...",[vouloir lever descendre enfant bras désireux ...


In [31]:
chemin_sortie = Path("..") /"data" /"4_resultats" /"informations_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
best_topic_model.get_topic_info().to_csv(chemin_sortie, index=False, encoding="utf-8")


## 7. Évaluation du modèle final

In [32]:
# L'analyseur produit exactement les tokens 
# attendus par le nouveau CountVectorizer.

analyseur_final = (vectorizer_final.build_analyzer())

texts_tokenises_final = [
    analyseur_final(document)
    for document in documents
]


dictionary_final = Dictionary(texts_tokenises_final)


coherence_finale = calculer_coherence_cv(
    topic_model=best_topic_model,
    labels=topics_finaux,
    texts_tokenises=texts_tokenises_final,
    dictionary=dictionary_final,
    top_n_words=10
)


diversite_finale = calculer_diversite_topics(
    topic_model=best_topic_model,
    labels=topics_finaux,
    top_n_words=10
)


print(
    f"Cohérence C_v finale : "
    f"{coherence_finale:.3f}"
)

print(
    f"Diversité finale : "
    f"{diversite_finale:.3f}"
)

Cohérence C_v finale : 0.389
Diversité finale : 0.947


## 8. Visualisations thématiques et temporelles

In [33]:
fig = best_topic_model.visualize_hierarchy()
fig.show()

In [34]:
# Calcul de l'évolution temporelle
topics_over_time = best_topic_model.topics_over_time(
    documents,
    df_model["annee"].tolist(),
    nr_bins=15
)

# Noms plus lisibles

# Création du graphique
fig = best_topic_model.visualize_topics_over_time(
    topics_over_time,
    custom_labels=True,
    normalize_frequency=False,
    title="Travail et milieux populaires",
    width=1500,
    height=700
)

# Personnalisation en français
fig.update_traces(mode="lines+markers")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Période de publication",
    yaxis_title="Nombre de segments",
    legend_title_text=None
)



15it [00:02,  6.79it/s]


## 9. Analyse de la distribution des topics par roman

In [35]:
df_resultats = df_model.copy()

df_resultats["topic"] = topics_finaux

df_resultats["est_outlier"] = (df_resultats["topic"] == -1)

display(df_resultats[[COL_ROMAN, COL_TEXTE, "topic", "est_outlier"]].head())

,roman,phrases_lemm,topic,est_outlier
0,1865 La confession de Claude.,voici hiver air matin devenir frais mettre man...,16,False
1,1865 La confession de Claude.,grenier haut escalier humide grand irrégulier ...,0,False
2,1865 La confession de Claude.,soir vent ébranler porte mur vaciller flamme l...,0,False
3,1865 La confession de Claude.,foyer demande grand feu joyeux vase oublier ne...,0,False
4,1865 La confession de Claude.,paraître déserte misérable vent pénétrer froid...,2,False


In [36]:
table_topics_romans_counts = pd.crosstab(df_resultats[COL_ROMAN],df_resultats["topic"])

display(table_topics_romans_counts)

topic,-1,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
roman,,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,0,79,11,278,2,1,68,9,35,21,6,49,38,6,0,7,6,33
1866 Le voeu d une morte.,0,25,5,16,5,32,53,13,143,84,3,26,58,10,5,9,18,14
1867 Les mysteres de Marseille.,0,45,27,36,211,194,131,29,112,152,23,32,121,225,66,131,139,25
1867 Therese Raquin.,0,41,6,76,10,43,228,33,159,95,12,48,4,9,15,10,16,122
1868 Madeleine Ferat.,0,86,9,188,5,44,201,27,363,107,21,76,43,23,14,7,35,91
Au Bonheur des dames.,1,315,41,35,41,250,105,304,113,487,126,88,51,29,27,108,32,15
Fecondite.,0,143,435,184,45,274,214,210,342,519,64,155,152,78,94,176,74,101
Germinal.,0,157,136,80,193,109,192,431,83,162,496,43,19,55,60,104,92,39
L argent.,2,131,207,30,48,498,65,94,117,260,102,66,45,114,82,170,111,6


In [37]:
table_topics_romans_proportions = pd.crosstab(
    df_resultats[COL_ROMAN],
    df_resultats["topic"],
    normalize="index"
)

display(table_topics_romans_proportions.style.format("{:.1%}"))

topic,-1,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
roman,,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,0.0%,12.2%,1.7%,42.8%,0.3%,0.2%,10.5%,1.4%,5.4%,3.2%,0.9%,7.6%,5.9%,0.9%,0.0%,1.1%,0.9%,5.1%
1866 Le voeu d une morte.,0.0%,4.8%,1.0%,3.1%,1.0%,6.2%,10.2%,2.5%,27.6%,16.2%,0.6%,5.0%,11.2%,1.9%,1.0%,1.7%,3.5%,2.7%
1867 Les mysteres de Marseille.,0.0%,2.6%,1.6%,2.1%,12.4%,11.4%,7.7%,1.7%,6.6%,8.9%,1.4%,1.9%,7.1%,13.2%,3.9%,7.7%,8.2%,1.5%
1867 Therese Raquin.,0.0%,4.4%,0.6%,8.2%,1.1%,4.6%,24.6%,3.6%,17.2%,10.2%,1.3%,5.2%,0.4%,1.0%,1.6%,1.1%,1.7%,13.2%
1868 Madeleine Ferat.,0.0%,6.4%,0.7%,14.0%,0.4%,3.3%,15.0%,2.0%,27.1%,8.0%,1.6%,5.7%,3.2%,1.7%,1.0%,0.5%,2.6%,6.8%
Au Bonheur des dames.,0.0%,14.5%,1.9%,1.6%,1.9%,11.5%,4.8%,14.0%,5.2%,22.5%,5.8%,4.1%,2.4%,1.3%,1.2%,5.0%,1.5%,0.7%
Fecondite.,0.0%,4.4%,13.3%,5.6%,1.4%,8.4%,6.6%,6.4%,10.5%,15.9%,2.0%,4.8%,4.7%,2.4%,2.9%,5.4%,2.3%,3.1%
Germinal.,0.0%,6.4%,5.5%,3.3%,7.9%,4.4%,7.8%,17.6%,3.4%,6.6%,20.2%,1.8%,0.8%,2.2%,2.4%,4.2%,3.8%,1.6%
L argent.,0.1%,6.1%,9.6%,1.4%,2.2%,23.2%,3.0%,4.4%,5.4%,12.1%,4.7%,3.1%,2.1%,5.3%,3.8%,7.9%,5.2%,0.3%


In [38]:
topics_par_roman = (
    best_topic_model
    .topics_per_class(
        documents,
        classes=romans,
        global_tuning=True
    )
)

display(topics_par_roman)

31it [00:01, 19.93it/s]


,Topic,Words,Frequency,Class
0,-1,"dormeur, indication, égorger, ,",1,Le ventre de Paris.
1,0,"chou, étalage, navet, volaille, manne",307,Le ventre de Paris.
2,1,"copier, manifeste, cayenne, minimum, liberticide",17,Le ventre de Paris.
3,2,"orge, quenu, resserre, battrai, justification",42,Le ventre de Paris.
4,3,"sergent, casemate, hollandais, écueil, île",42,Le ventre de Paris.
...,...,...,...,...
531,12,"piller, cabaretier, mouchard, régie, feignant",55,Germinal.
532,13,"terrier, baril, érafler, dénoncer, grève",60,Germinal.
533,14,"centime, berline, boisage, tarif, grève",104,Germinal.
534,15,"grève, popularité, haveur, deneulin, régisseur",92,Germinal.


## 10. Export des résultats

In [39]:
chemin_sortie = Path("..") /"data" /"4_resultats" /"documents_avec_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
df_resultats.to_csv(chemin_sortie, index=False, encoding="utf-8")


chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_comptages.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_counts.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_proportions.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_proportions.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"bertopic_topics_per_class.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
topics_par_roman.to_csv(chemin_sortie, index=False, encoding="utf-8")



